<a href="https://colab.research.google.com/github/sandeepgangaram/neural-nets/blob/master/makemor/makemor_mlp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [7]:
words = open('./names.txt','r').read().splitlines()
words[:5]

FileNotFoundError: [Errno 2] No such file or directory: './names.txt'

In [ ]:
len(words)

In [ ]:
from IPython.core.crashhandler import crash_handler_lite
#build the vocabulary of characters and mappings to/from integers
chars = sorted(set(''.join(words)))
stoi = {char:i+1 for i,char in enumerate(chars)}
stoi['.'] = 0
itos = {i: char for char, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)
# itos

In [ ]:
block_size = 3 #context length - characters to predict next character

# build the dataset
def build_dataset(words):
  X, Y = [],[]

  for w in words:
    # print(w)
    context = [0]*block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      # print(''.join(itos[i] for i in context),'--->', itos[ix])
      context = context[1:] + [ix]
  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape,Y.shape)
  return X,Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])     #training set 80% - train parameters
Xdev, Ydev = build_dataset(words[n1:n2]) #dev/validation set 10% - train hyperparameters
Xte, Yte = build_dataset(words[n2:])     #test set 10%


In [ ]:
# putting it all together
n_embed = 10
n_hidden = 200
batch_size = 32
max_steps = 200000

g = torch.Generator().manual_seed(2147483647)

#input layer
C = torch.randn(vocab_size,n_embed, generator=g)

# hidden layer
W1 = torch.rand((n_embed*block_size, n_hidden), generator=g)*(5/3)/(n_embed*block_size)**0.5
b1 = torch.randn(n_hidden, generator=g)*0.1

#output layer
W2 = torch.randn((n_hidden,vocab_size), generator=g)*0.1
b2 = torch.rand(vocab_size, generator=g)*0

###
# batch norm parameters - learnable parameters
#  (used to keep guassian only on init and let network
# flow with the learning instaed of always constraining to a guassion)
###
bngain = torch.ones(1, n_hidden)
bnbias = torch.zeros(1, n_hidden)

#these are used at inference allowing to forward individual examples
bnmean_running = torch.zeros(1, n_hidden)
bnstd_running = torch.ones(1, n_hidden)


parameters = [C,W1, W2, b1, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
  p.requires_grad = True

In [ ]:
x = torch.randn(100,10)
w = torch.randn(10,200)
y = x@w
print(x.mean(), x.std(), x.var())
print(w.mean(), w.std(), w.var())
print(y.mean(), y.std(), y.var())
plt.figure(figsize=(20,5))
plt.subplot(121)
plt.hist(x.view(-1).tolist(),50, density=True)
plt.subplot(122)
plt.hist(y.view(-1).tolist(),50, density=True)

In [ ]:
# hpreact.mean(0,keepdim=True).shape
# hpreact.std(0,keepdim=True).shape

In [ ]:

lossi = []

for i in range(max_steps):
  #mini-batch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb, Yb = Xtr[ix], Ytr[ix]

  #forward-pass
  emb = C[Xb]
  embcat = emb.view(emb.shape[0],-1)
  hpreact = embcat@W1 + #b1 #hidden-layer preactiation (bias not needed in batch norm.)

  #batch normalization step
  bnmeani = hpreact.mean(0,keepdim=True)
  bnstdi = hpreact.std(0, keepdim=True)
  hpreact = bngain*((hpreact - bnmeani)/bnstdi) + bnbias

  with torch.no_grad():
    bnmean_running = 0.999*bnmean_running + 0.001*bnmeani
    bnstd_running = 0.999*bnstd_running + 0.001*bnstdi

  h = torch.tanh(hpreact)
  logits = h@W2 + b2
  loss = F.cross_entropy(logits,Yb)

  #backward-pass
  for p in parameters:
    p.grad = None
  loss.backward()

  #learning reate decay - once loss plateus, reduce the learning rate /10
  lr = 0.1 if i < 100000 else 0.01

  for p in parameters:
    p.data -= lr*p.grad

  # print once in a while
  if i%10000 == 0:
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())
  # break

In [ ]:
# plt.hist(h.view(-1).tolist(),50)

In [ ]:
# plt.hist(hpreact.view(-1).tolist(),50)

In [ ]:
# plt.figure(figsize=(20,10))
# plt.imshow(h.abs() > 0.99, cmap='gray', interpolation='nearest')

In [ ]:
# logits[0]

In [ ]:
# logits = torch.tensor([1.0,2.0,1.0,1.0])
# probs = F.softmax(logits, dim=0)
# loss = -probs[2].log()
# logits,probs,loss

In [ ]:
# -torch.tensor(1/27.0).log()

In [ ]:
plt.plot(lossi)

In [ ]:
with torch.no_grad():
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0],-1)
  hpreact = embcat@W1 + b1
  #measure the mean/std. over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnstd = hpreact.std(0, keepdim=True)

In [ ]:
bnstd_running

In [ ]:
bnstd

In [ ]:
@torch.no_grad() #decorator to disable gradient tracking - not required here
def split_loss(split):
  x,y = {
      'train':(Xtr,Ytr),
      'val':(Xdev,Ydev),
      'test':(Xte,Yte)
  }[split]
  emb = C[x]
  embcat = emb.view(emb.shape[0],-1)
  hpreact = embcat@W1 + b1
  #batch normalization step
  hpreact = bngain*((hpreact - bnmean_running)/bnstd_running) + bnbias
  h = torch.tanh(hpreact)
  logits = h@W2 + b2
  loss = F.cross_entropy(logits,y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

In [ ]:

#before - random initialization
# train 2.12
# val 2.16

##1
# fix softmax being confidently wrong -

# by changing initial output weights and biases bringing them closer to 0
# w2 * 0.1
# b2*0

# train 2.044
# val 2.11

##2 fix tanh layer too saturated at init
# train 2.05
# val 2.10

## use semi-principled "Kaiming" initalization instead of hacky init
# train 2.0452935695648193
# val 2.107194423675537

## add a batch normalization layer
# train 2.067406415939331
# val 2.10783314704895


In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647)

for _ in range(20):

    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break

    print(''.join(itos[i] for i in out))


In [ ]:
#batch noramalization is used to control the statistics of the activations in the neural net
#typiclly plced after a multiplication layer (ex: linear or convolutional layer)
#has parameters gain and bias that are also trained using back-propogation
#has buffers - mean and std. deviation - updated on each step